# Baseline climatologica - variaveis fisicas com gold

Este notebook cria uma baseline deterministica para prever irradiacao solar diaria e velocidade media do vento usando apenas historico da camada gold. A geracao em kWh e calculada depois da predicao fisica.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import mlflow.sklearn
import pandas as pd

from src.modeling.gold_energy import (
    ENERGY_RESULT_COLUMNS,
    TARGET_COLUMNS,
    build_prediction_results_table,
    calculate_energy_outputs_from_physical,
    clip_physical_predictions,
    configure_mlflow_tracking,
    evaluate_predictions,
    load_gold_daily,
    make_future_feature_frame,
    prepare_energy_modeling_table,
    split_feature_target_metadata,
    temporal_train_test_split,
    to_jsonable,
)
from src.modeling.historical_features import (
    ClimatologyBaselineRegressor,
    fit_historical_feature_reference,
    predict_climatology_baseline,
    save_historical_reference,
)
from src.modeling.training_config import (
    BASELINE_MODEL_NAME,
    ENERGY_CONFIG,
    FUTURE_DATE,
    FUTURE_STATION_CODE,
    HISTORY_MIN_OBSERVATIONS_DAY,
    HISTORY_MIN_OBSERVATIONS_MONTH,
    MLFLOW_EXPERIMENT_NAME,
    PRODUCTION_REFIT_WITH_FULL_GOLD,
    TEST_YEAR_FRACTION,
)

RUN_ARTIFACTS_DIR = configure_mlflow_tracking(PROJECT_ROOT, MLFLOW_EXPERIMENT_NAME)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Diretorio de artefatos de modelagem: {RUN_ARTIFACTS_DIR}")

C:\Users\Admin\Desktop\Projetos\Projeto-ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/06/13 13:46:06 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/06/13 13:46:06 INFO mlflow.store.db.utils: Updating database tables


Raiz do projeto: C:\Users\Admin\Desktop\Projetos\Projeto-ML
Diretorio de artefatos de modelagem: C:\Users\Admin\Desktop\Projetos\Projeto-ML\artifacts\modeling\20260613-134558


In [2]:
# Configuracoes da baseline ficam em src/modeling/training_config.py.
MODEL_NAME = BASELINE_MODEL_NAME

assert HISTORY_MIN_OBSERVATIONS_DAY > 0, "HISTORY_MIN_OBSERVATIONS_DAY deve ser positivo."
assert HISTORY_MIN_OBSERVATIONS_MONTH > 0, "HISTORY_MIN_OBSERVATIONS_MONTH deve ser positivo."
print(f"HISTORY_MIN_OBSERVATIONS_DAY={HISTORY_MIN_OBSERVATIONS_DAY}")
print(f"HISTORY_MIN_OBSERVATIONS_MONTH={HISTORY_MIN_OBSERVATIONS_MONTH}")
print(f"PRODUCTION_REFIT_WITH_FULL_GOLD={PRODUCTION_REFIT_WITH_FULL_GOLD}")

HISTORY_MIN_OBSERVATIONS_DAY=3
HISTORY_MIN_OBSERVATIONS_MONTH=3
PRODUCTION_REFIT_WITH_FULL_GOLD=True


In [3]:
daily_gold = load_gold_daily(PROJECT_ROOT)
modeling_table = prepare_energy_modeling_table(daily_gold, ENERGY_CONFIG)
X, y, metadata = split_feature_target_metadata(modeling_table)

(
    X_train,
    X_test,
    y_train,
    y_test,
    train_metadata,
    test_metadata,
    train_years,
    test_years,
) = temporal_train_test_split(X, y, metadata, test_year_fraction=TEST_YEAR_FRACTION)

baseline_model = ClimatologyBaselineRegressor(
    min_observations_day=HISTORY_MIN_OBSERVATIONS_DAY,
    min_observations_month=HISTORY_MIN_OBSERVATIONS_MONTH,
)
X_train_for_baseline = X_train.copy()
X_train_for_baseline["year"] = train_metadata["year"].to_numpy()
baseline_model.fit(X_train_for_baseline, y_train)
train_reference = baseline_model.reference_
production_reference = (
    fit_historical_feature_reference(
        modeling_table,
        min_observations_day=HISTORY_MIN_OBSERVATIONS_DAY,
        min_observations_month=HISTORY_MIN_OBSERVATIONS_MONTH,
    )
    if PRODUCTION_REFIT_WITH_FULL_GOLD
    else train_reference
)

test_predictions = clip_physical_predictions(baseline_model.predict(X_test))
metrics_table, final_metrics = evaluate_predictions(y_test, test_predictions)
test_results, energy_actual, energy_pred, _ = build_prediction_results_table(
    test_metadata,
    y_test,
    test_predictions,
    ENERGY_CONFIG,
)
energy_metrics_table, energy_metrics = evaluate_predictions(
    energy_actual[ENERGY_RESULT_COLUMNS],
    energy_pred[ENERGY_RESULT_COLUMNS],
    target_columns=ENERGY_RESULT_COLUMNS,
)

print(f"Linhas gold usadas: {len(modeling_table):,}")
print(f"Alvos fisicos do ML: {TARGET_COLUMNS}")
print(f"Saidas energeticas calculadas: {ENERGY_RESULT_COLUMNS}")
print(f"Anos treino/validacao: {train_years}")
print(f"Anos teste final: {test_years}")
print("Metricas fisicas da baseline no teste temporal final:")
display(metrics_table)
print("Metricas energeticas calculadas da baseline:")
display(energy_metrics_table)

Linhas gold usadas: 58,022
Alvos fisicos do ML: ['solar_daily_kwh_m2_day', 'wind_daily_mean_ms']
Saidas energeticas calculadas: ['solar_generation_kwh_day', 'wind_generation_kwh_day', 'hybrid_generation_kwh_day']
Anos treino/validacao: [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
Anos teste final: [2021, 2022, 2023, 2024, 2025]
Metricas fisicas da baseline no teste temporal final:


,target,mae,rmse,nrmse,r2,bias,medae,smape
0,solar_daily_kwh_m2_day,1.340608,1.696688,0.164762,-0.113885,0.150524,1.127885,27.182303
1,wind_daily_mean_ms,0.873305,1.174111,0.176411,-0.234772,0.411390,0.650388,38.179938


Metricas energeticas calculadas da baseline:


,target,mae,rmse,nrmse,r2,bias,medae,smape
0,solar_generation_kwh_day,233.547481,295.580117,0.164762,-0.113885,26.222758,196.488936,27.182303
1,wind_generation_kwh_day,4297.198228,6415.694604,0.095312,-0.120071,840.685409,3015.222139,81.562432
2,hybrid_generation_kwh_day,4333.153921,6450.438357,0.094740,-0.126437,866.908166,3039.771925,65.784502


In [4]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name=f"{MODEL_NAME}_physical_gold"):
    mlflow_run_id = mlflow.active_run().info.run_id
    mlflow.log_param("model_family", "ClimatologyBaselineRegressor")
    mlflow.log_param("train_years", ",".join(map(str, train_years)))
    mlflow.log_param("test_years", ",".join(map(str, test_years)))
    mlflow.log_param("targets", ",".join(TARGET_COLUMNS))
    mlflow.log_param("energy_outputs", ",".join(ENERGY_RESULT_COLUMNS))
    mlflow.log_param("history_min_observations_day", HISTORY_MIN_OBSERVATIONS_DAY)
    mlflow.log_param("history_min_observations_month", HISTORY_MIN_OBSERVATIONS_MONTH)
    mlflow.log_param("production_refit_with_full_gold", PRODUCTION_REFIT_WITH_FULL_GOLD)
    mlflow.log_dict(to_jsonable(ENERGY_CONFIG), "energy_config.json")
    mlflow.log_dict(to_jsonable(train_reference.metadata()), "historical_reference_train_metadata.json")
    mlflow.log_dict(to_jsonable(production_reference.metadata()), "historical_reference_production_metadata.json")
    for metric_name, metric_value in final_metrics.items():
        if pd.notna(metric_value):
            mlflow.log_metric(f"physical_{metric_name}", float(metric_value))
    for metric_name, metric_value in energy_metrics.items():
        if pd.notna(metric_value):
            mlflow.log_metric(f"energy_{metric_name}", float(metric_value))
    mlflow.sklearn.log_model(baseline_model, artifact_path="model")

print("RESULTADOS FINAIS - BASELINE CLIMATOLOGICA")
print(f"MLflow run_id: {mlflow_run_id}")
print(f"Anos de treino/validacao: {train_years}")
print(f"Anos de teste final: {test_years}")
display(metrics_table)
display(energy_metrics_table)

2026/06/13 13:46:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/06/13 13:46:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RESULTADOS FINAIS - BASELINE CLIMATOLOGICA
MLflow run_id: 3b78e07ba36145d4a014f793f1daecdc
Anos de treino/validacao: [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
Anos de teste final: [2021, 2022, 2023, 2024, 2025]


,target,mae,rmse,nrmse,r2,bias,medae,smape
0,solar_daily_kwh_m2_day,1.340608,1.696688,0.164762,-0.113885,0.150524,1.127885,27.182303
1,wind_daily_mean_ms,0.873305,1.174111,0.176411,-0.234772,0.411390,0.650388,38.179938


,target,mae,rmse,nrmse,r2,bias,medae,smape
0,solar_generation_kwh_day,233.547481,295.580117,0.164762,-0.113885,26.222758,196.488936,27.182303
1,wind_generation_kwh_day,4297.198228,6415.694604,0.095312,-0.120071,840.685409,3015.222139,81.562432
2,hybrid_generation_kwh_day,4333.153921,6450.438357,0.094740,-0.126437,866.908166,3039.771925,65.784502


In [5]:
# Resumo explicito das metricas fisicas e energeticas no teste temporal final.
print("RESUMO DAS METRICAS - TESTE TEMPORAL FINAL")
print(f"Modelo: {MODEL_NAME}")
print(f"MLflow run_id: {mlflow_run_id}")
print(f"Anos de teste final: {test_years}")
print(f"physical_balanced_nrmse: {final_metrics['balanced_nrmse']:.6f}")

print("")
print("Metricas fisicas previstas pelo ML:")
for _, row in metrics_table.iterrows():
    print(f"Alvo: {row['target']}")
    print(f"  MAE: {row['mae']:.6f}")
    print(f"  RMSE: {row['rmse']:.6f}")
    print(f"  NRMSE: {row['nrmse']:.6f}")
    print(f"  R2: {row['r2']:.6f}")
    print(f"  Bias medio: {row['bias']:.6f}")
    print(f"  MedAE: {row['medae']:.6f}")
    print(f"  sMAPE (%): {row['smape']:.6f}")
print("")
print("Metricas energeticas calculadas apos a predicao fisica:")
for _, row in energy_metrics_table.iterrows():
    print(f"Saida: {row['target']}")
    print(f"  MAE: {row['mae']:.6f}")
    print(f"  RMSE: {row['rmse']:.6f}")
    print(f"  NRMSE: {row['nrmse']:.6f}")
    print(f"  R2: {row['r2']:.6f}")
    print(f"  Bias medio: {row['bias']:.6f}")
    print(f"  MedAE: {row['medae']:.6f}")
    print(f"  sMAPE (%): {row['smape']:.6f}")

RESUMO DAS METRICAS - TESTE TEMPORAL FINAL
Modelo: baseline_climatology
MLflow run_id: 3b78e07ba36145d4a014f793f1daecdc
Anos de teste final: [2021, 2022, 2023, 2024, 2025]
physical_balanced_nrmse: 0.170586

Metricas fisicas previstas pelo ML:
Alvo: solar_daily_kwh_m2_day
  MAE: 1.340608
  RMSE: 1.696688
  NRMSE: 0.164762
  R2: -0.113885
  Bias medio: 0.150524
  MedAE: 1.127885
  sMAPE (%): 27.182303
Alvo: wind_daily_mean_ms
  MAE: 0.873305
  RMSE: 1.174111
  NRMSE: 0.176411
  R2: -0.234772
  Bias medio: 0.411390
  MedAE: 0.650388
  sMAPE (%): 38.179938

Metricas energeticas calculadas apos a predicao fisica:
Saida: solar_generation_kwh_day
  MAE: 233.547481
  RMSE: 295.580117
  NRMSE: 0.164762
  R2: -0.113885
  Bias medio: 26.222758
  MedAE: 196.488936
  sMAPE (%): 27.182303
Saida: wind_generation_kwh_day
  MAE: 4297.198228
  RMSE: 6415.694604
  NRMSE: 0.095312
  R2: -0.120071
  Bias medio: 840.685409
  MedAE: 3015.222139
  sMAPE (%): 81.562432
Saida: hybrid_generation_kwh_day
  MAE: 4

In [6]:
results_dir = RUN_ARTIFACTS_DIR / "evaluation" / MODEL_NAME
results_dir.mkdir(parents=True, exist_ok=True)
run_timestamp = pd.Timestamp.now().strftime("%Y%m%d-%H%M%S")
metrics_output_path = results_dir / f"{MODEL_NAME}_physical_metrics_{run_timestamp}.csv"
energy_metrics_output_path = results_dir / f"{MODEL_NAME}_energy_metrics_{run_timestamp}.csv"
metrics_json_output_path = results_dir / f"{MODEL_NAME}_physical_metrics_{run_timestamp}.json"
energy_metrics_json_output_path = results_dir / f"{MODEL_NAME}_energy_metrics_{run_timestamp}.json"
predictions_output_path = results_dir / f"{MODEL_NAME}_test_predictions_{run_timestamp}.csv"
predictions_sample_output_path = results_dir / f"{MODEL_NAME}_test_predictions_sample_{run_timestamp}.csv"
train_reference_dir = results_dir / "historical_reference_train"
production_reference_dir = results_dir / "historical_reference_production"

metrics_table.to_csv(metrics_output_path, index=False)
energy_metrics_table.to_csv(energy_metrics_output_path, index=False)
pd.Series(final_metrics).to_json(metrics_json_output_path, indent=2)
pd.Series(energy_metrics).to_json(energy_metrics_json_output_path, indent=2)
test_results.to_csv(predictions_output_path, index=False)
test_results.head(50).to_csv(predictions_sample_output_path, index=False)
save_historical_reference(train_reference, train_reference_dir)
save_historical_reference(production_reference, production_reference_dir)

with mlflow.start_run(run_id=mlflow_run_id):
    mlflow.log_artifact(str(metrics_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(energy_metrics_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(metrics_json_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(energy_metrics_json_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(predictions_sample_output_path), artifact_path="evaluation")
    mlflow.log_artifacts(str(train_reference_dir), artifact_path="historical_reference_train")
    mlflow.log_artifacts(str(production_reference_dir), artifact_path="historical_reference_production")

print("Arquivos de resultados salvos:")
print(f"- Metricas fisicas CSV: {metrics_output_path}")
print(f"- Metricas energeticas CSV: {energy_metrics_output_path}")
print(f"- Predicoes completas do teste: {predictions_output_path}")
print(f"- Amostra das predicoes: {predictions_sample_output_path}")
print(f"- Referencia historica de treino: {train_reference_dir}")
print(f"- Referencia historica de producao: {production_reference_dir}")
print("Primeiras predicoes do teste temporal:")
display(test_results.head(20))

Arquivos de resultados salvos:
- Metricas fisicas CSV: C:\Users\Admin\Desktop\Projetos\Projeto-ML\artifacts\modeling\20260613-134558\evaluation\baseline_climatology\baseline_climatology_physical_metrics_20260613-134616.csv
- Metricas energeticas CSV: C:\Users\Admin\Desktop\Projetos\Projeto-ML\artifacts\modeling\20260613-134558\evaluation\baseline_climatology\baseline_climatology_energy_metrics_20260613-134616.csv
- Predicoes completas do teste: C:\Users\Admin\Desktop\Projetos\Projeto-ML\artifacts\modeling\20260613-134558\evaluation\baseline_climatology\baseline_climatology_test_predictions_20260613-134616.csv
- Amostra das predicoes: C:\Users\Admin\Desktop\Projetos\Projeto-ML\artifacts\modeling\20260613-134558\evaluation\baseline_climatology\baseline_climatology_test_predictions_sample_20260613-134616.csv
- Referencia historica de treino: C:\Users\Admin\Desktop\Projetos\Projeto-ML\artifacts\modeling\20260613-134558\evaluation\baseline_climatology\historical_reference_train
- Referencia

,station_code,station_name,city,state,latitude,longitude,altitude,date,year,solar_daily_kwh_m2_day_actual,...,wind_daily_mean_ms_actual,wind_daily_mean_ms_pred,wind_daily_mean_hub_height_ms_actual,wind_daily_mean_hub_height_ms_pred,solar_generation_kwh_day_actual,solar_generation_kwh_day_pred,wind_generation_kwh_day_actual,wind_generation_kwh_day_pred,hybrid_generation_kwh_day_actual,hybrid_generation_kwh_day_pred
0,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-01,2021,6.988250,...,1.754167,1.758013,2.807429,2.813584,1217.423766,1051.891781,1232.406026,1240.530266,2449.829792,2292.422047
1,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-02,2021,6.179944,...,1.808333,1.813542,2.894119,2.902454,1076.608771,1107.390325,1350.133503,1361.833035,2426.742274,2469.223360
2,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-03,2021,1.452278,...,0.912500,1.679156,1.460396,2.687379,253.001464,1198.368153,173.476188,1080.972466,426.477652,2279.340619
3,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-04,2021,6.525583,...,1.087500,1.670563,1.740472,2.673626,1136.822558,1016.234945,293.649353,1064.460737,1430.471910,2080.695682
4,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-05,2021,5.342833,...,1.137500,1.780952,1.820494,2.850297,930.775556,1081.519836,336.043475,1289.728133,1266.819031,2371.247969
5,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-06,2021,5.760806,...,0.850000,1.691964,1.360369,2.707878,1003.590541,1128.483977,140.216147,1105.897371,1143.806687,2234.381348
6,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-07,2021,3.299306,...,1.343478,1.576190,2.150149,2.522589,574.772367,1082.298251,553.647293,894.061406,1128.419661,1976.359657
7,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-08,2021,4.362028,...,2.316667,1.663242,3.707673,2.661910,759.909317,1044.573814,2838.781069,1050.528093,3598.690386,2095.101906
8,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-09,2021,5.155222,...,1.566667,1.586538,2.507347,2.539151,898.091805,966.991529,877.952582,911.786307,1776.044387,1878.777835
9,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-10,2021,6.972861,...,1.408333,1.860543,2.253945,2.977677,1214.742866,1026.010817,637.760513,1470.484112,1852.503380,2496.494928


In [7]:
# Preencha FUTURE_STATION_CODE e FUTURE_DATE em src/modeling/training_config.py.
if FUTURE_STATION_CODE is None or FUTURE_DATE is None:
    print("Preencha FUTURE_STATION_CODE e FUTURE_DATE para gerar inferencia futura.")
else:
    future_frame = make_future_feature_frame(modeling_table, FUTURE_DATE)
    future_predictions = clip_physical_predictions(predict_climatology_baseline(future_frame, production_reference))
    future_energy = calculate_energy_outputs_from_physical(future_predictions, ENERGY_CONFIG, include_hub_height=True)
    future_ranking = future_frame[[
        column
        for column in ["station_code", "station_name", "city", "state", "latitude", "longitude", "altitude", "date"]
        if column in future_frame.columns
    ]].copy()
    for column in TARGET_COLUMNS:
        future_ranking[f"{column}_pred"] = future_predictions[column].to_numpy()
    for column in future_energy.columns:
        future_ranking[f"{column}_pred"] = future_energy[column].to_numpy()
    future_ranking = future_ranking.sort_values("hybrid_generation_kwh_day_pred", ascending=False).reset_index(drop=True)

    station_code = str(FUTURE_STATION_CODE)
    station_prediction = future_ranking[future_ranking["station_code"].astype(str) == station_code].copy()
    if station_prediction.empty:
        known = ", ".join(future_ranking["station_code"].astype(str).head(10).tolist())
        raise ValueError(f"station_code nao encontrado na gold: {station_code}. Exemplos conhecidos: {known}")
    station_prediction.insert(0, "ranking_position", station_prediction.index + 1)
    display(station_prediction.reset_index(drop=True))
    display(future_ranking.head(20))

Preencha FUTURE_STATION_CODE e FUTURE_DATE para gerar inferencia futura.
